# 11.02 -- Tensor Parallelism and Distributed Inference

**Module:** 11 -- LLM Systems Engineering  
**Notebook:** 2 of 5  
**Prerequisites:** 11.01 (GPU Memory Planning)

---

## Learning Objectives

1. Understand tensor parallelism: splitting weight matrices across devices
2. Implement column-parallel and row-parallel linear layers from scratch
3. Understand pipeline parallelism and its trade-off with tensor parallelism
4. Measure communication overhead for all-reduce vs point-to-point
5. Understand how vLLM and Megatron-LM implement parallelism for production inference

---

## 1. Why Parallelism Is Necessary

A single A100-80GB GPU holds models up to roughly 35 B parameters in FP16 (with a batch).
Models above this size require distributing computation and memory across multiple GPUs.
There are three primary parallelism strategies for LLM inference:

```
Strategy          What is split          Communication        Latency impact
----------        -------------------    -------------------  ---------------
Tensor (TP)       Each weight matrix     All-reduce per layer  Low (< 1ms NVLink)
Pipeline (PP)     Groups of layers       P2P between stages    Medium (bubble overhead)
Sequence (SP)     Token sequence itself  All-gather per layer  Low (recent method)
```

For interactive inference, tensor parallelism is preferred because it keeps all GPUs
busy simultaneously (no pipeline bubbles). Pipeline parallelism is preferred for
high-throughput batch workloads where the pipeline can be kept full.

---

## 2. Tensor Parallelism: The Mathematics

A standard linear layer computes `Y = X W^T` where X is (B, in) and W is (out, in).

**Column parallelism** splits W along the output dimension:
```
  W = [W_0 | W_1 | ... | W_{N-1}]  (each has shape (out/N, in))
  Y_i = X W_i^T   on GPU i          (shape: B, out/N)
  Y = concat(Y_0, ..., Y_{N-1})     (all-gather, final shape: B, out)
```

**Row parallelism** splits W along the input dimension:
```
  W = [W_0; W_1; ...; W_{N-1}]     (each has shape (out, in/N))
  X_i = X[:, i*in/N : (i+1)*in/N]  on GPU i
  Y_i = X_i W_i^T                   (shape: B, out)
  Y = sum(Y_0, ..., Y_{N-1})        (all-reduce, final shape: B, out)
```

In transformer attention, the Q/K/V projections use column parallelism
and the output projection uses row parallelism. This pattern requires exactly
one all-reduce per attention layer and one per MLP layer.


In [ ]:
# Environment setup.
#
# We simulate tensor parallelism on CPU using regular Python/PyTorch tensors
# split across a list of 'virtual GPUs'. Each element of the list represents
# one device's shard of the computation. For multi-GPU demonstrations we
# use torch.distributed with NCCL when available.
#
# The simulation captures all the key mechanics: weight sharding, partial
# matrix multiplications, and the all-reduce / all-gather collectives.

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from typing import List, Tuple, Dict
import time
import warnings
warnings.filterwarnings('ignore')

torch.manual_seed(42)
np.random.seed(42)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
N_GPUS = torch.cuda.device_count() if torch.cuda.is_available() else 1
print(f'Device: {device}')
print(f'Available GPUs: {N_GPUS}')
print()
print('This notebook simulates multi-GPU tensor parallelism using sharded tensors.')
print('All correctness properties hold regardless of physical GPU count.')


## 3. Column-Parallel and Row-Parallel Linear Layers


In [ ]:
# Column-parallel and row-parallel linear layers from scratch.
#
# These are the two building blocks of tensor-parallel transformers.
# Megatron-LM, vLLM, and all modern LLM frameworks use exactly these
# two primitives to parallelise attention and MLP layers.
#
# ColumnParallelLinear:
#   Each device holds columns [i*out_per_rank : (i+1)*out_per_rank] of W.
#   Input X is broadcast to all devices (or each device already has X).
#   Each device computes a partial Y_i = X W_i^T.
#   Results are concatenated across devices: Y = cat(Y_0, ..., Y_{N-1}).
#
# RowParallelLinear:
#   Each device holds rows [i*in_per_rank : (i+1)*in_per_rank] of W.
#   Input X is assumed to already be split (comes from a ColumnParallelLinear).
#   Each device computes partial Y_i = X_i W_i^T.
#   Results are summed: Y = Y_0 + Y_1 + ... + Y_{N-1}  (all-reduce).

class ColumnParallelLinear(nn.Module):
    """
    Linear layer with output dimension split across N virtual ranks.

    Simulates column parallelism: each rank owns 1/N of the output features.
    In production, each rank runs on a separate GPU with NCCL for the all-gather.
    Here we hold all shards in a list on the same device for simulation.
    """

    def __init__(self, in_features: int, out_features: int, n_ranks: int, bias: bool = True):
        super().__init__()
        assert out_features % n_ranks == 0, 'out_features must be divisible by n_ranks'
        self.in_features    = in_features
        self.out_features   = out_features
        self.n_ranks        = n_ranks
        self.out_per_rank   = out_features // n_ranks

        # Each rank gets a shard of the weight matrix: shape (out/N, in)
        self.weight_shards = nn.ParameterList([
            nn.Parameter(torch.randn(self.out_per_rank, in_features) / math.sqrt(in_features))
            for _ in range(n_ranks)
        ])
        if bias:
            self.bias_shards = nn.ParameterList([
                nn.Parameter(torch.zeros(self.out_per_rank))
                for _ in range(n_ranks)
            ])
        else:
            self.bias_shards = None

    def forward(self, x: torch.Tensor) -> List[torch.Tensor]:
        """
        Each rank computes its partial output independently.
        Returns a list of N shards, each of shape (B, out/N).
        In production, each rank returns its local shard; the all-gather
        is performed by the communication layer after all ranks complete.
        """
        outputs = []
        for i in range(self.n_ranks):
            y_i = x @ self.weight_shards[i].T
            if self.bias_shards is not None:
                y_i = y_i + self.bias_shards[i]
            outputs.append(y_i)
        return outputs   # list of (B, out/N) tensors

    def forward_and_gather(self, x: torch.Tensor) -> torch.Tensor:
        """Convenience: compute all shards and concatenate (simulated all-gather)."""
        return torch.cat(self.forward(x), dim=-1)   # (B, out)


class RowParallelLinear(nn.Module):
    """
    Linear layer with input dimension split across N virtual ranks.

    Simulates row parallelism: each rank receives a pre-split input shard
    and computes a partial output. Results are summed (all-reduce).
    """

    def __init__(self, in_features: int, out_features: int, n_ranks: int, bias: bool = True):
        super().__init__()
        assert in_features % n_ranks == 0
        self.n_ranks       = n_ranks
        self.in_per_rank   = in_features // n_ranks

        # Each rank gets a shard of W: shape (out, in/N)
        self.weight_shards = nn.ParameterList([
            nn.Parameter(torch.randn(out_features, self.in_per_rank) / math.sqrt(in_features))
            for _ in range(n_ranks)
        ])
        self.bias = nn.Parameter(torch.zeros(out_features)) if bias else None

    def forward(self, x_shards: List[torch.Tensor]) -> torch.Tensor:
        """
        Each rank computes a partial output from its input shard.
        The partial outputs are summed (simulated all-reduce).

        x_shards: list of N tensors each of shape (B, in/N).
        Returns: (B, out) -- the full output after all-reduce.
        """
        partial_outputs = [
            x_shards[i] @ self.weight_shards[i].T
            for i in range(self.n_ranks)
        ]
        y = sum(partial_outputs)   # all-reduce = sum of partial results
        if self.bias is not None:
            y = y + self.bias
        return y


import math

# Verify correctness: column-parallel + row-parallel should equal a single linear layer
IN, OUT, N_RANKS, BATCH = 512, 512, 4, 8

col_layer = ColumnParallelLinear(IN, OUT, N_RANKS)
row_layer = RowParallelLinear(OUT, OUT, N_RANKS)

# Build equivalent single linear by assembling shards
W_col = torch.cat([s.data for s in col_layer.weight_shards], dim=0)  # (OUT, IN)
b_col = torch.cat([b.data for b in col_layer.bias_shards], dim=0)    # (OUT,)
W_row = torch.cat([s.data for s in row_layer.weight_shards], dim=1)  # (OUT, OUT)

x = torch.randn(BATCH, IN)

# Parallel path
mid_shards   = col_layer.forward(x)             # list of (B, OUT/4)
out_parallel = row_layer.forward(mid_shards)    # (B, OUT)

# Serial path
mid_serial   = x @ W_col.T + b_col
out_serial   = mid_serial @ W_row.T + row_layer.bias

max_err = (out_parallel - out_serial).abs().max().item()
print(f'Column + Row parallel correctness check:')
print(f'  Max absolute difference vs serial: {max_err:.2e}  (should be < 1e-5)')
print(f'  Verified: {max_err < 1e-4}')


In [ ]:
# Tensor-parallel attention head: splitting Q, K, V projections.
#
# In a tensor-parallel transformer, each GPU owns a subset of attention heads.
# With N_RANKS GPUs, each GPU runs n_heads / N_RANKS heads independently.
# The attention computation is fully local (no communication needed during attention).
# Communication only happens at the output projection (row-parallel all-reduce).
#
# This is the Megatron-LM parallelism pattern (Shoeybi et al., 2019):
#   Q, K, V projections:  column-parallel (each GPU computes its heads)
#   Attention:            local (no communication)
#   Output projection:    row-parallel (all-reduce to aggregate heads)
#   MLP FC1:              column-parallel
#   MLP FC2:              row-parallel
#
# Communication count: 2 all-reduces per transformer layer (one after attention,
# one after MLP). On NVLink (600 GB/s), each all-reduce for a 4096-dim hidden
# state at batch=8 costs roughly 0.05 ms. For a 32-layer model: ~3 ms total.

class TensorParallelAttention(nn.Module):
    """
    Multi-head attention with tensor parallelism across n_ranks.

    Each rank owns n_heads // n_ranks attention heads.
    Computation follows the Megatron-LM pattern.
    """

    def __init__(self, hidden_size: int, n_heads: int, n_ranks: int):
        super().__init__()
        assert n_heads % n_ranks == 0
        self.hidden_size    = hidden_size
        self.n_heads        = n_heads
        self.n_ranks        = n_ranks
        self.heads_per_rank = n_heads // n_ranks
        self.head_dim       = hidden_size // n_heads

        # Column-parallel: each rank owns heads_per_rank * head_dim output features
        out_per_rank = self.heads_per_rank * self.head_dim
        self.q_proj  = ColumnParallelLinear(hidden_size, hidden_size, n_ranks, bias=False)
        self.k_proj  = ColumnParallelLinear(hidden_size, hidden_size, n_ranks, bias=False)
        self.v_proj  = ColumnParallelLinear(hidden_size, hidden_size, n_ranks, bias=False)

        # Row-parallel: reduces partial outputs from all ranks
        self.out_proj = RowParallelLinear(hidden_size, hidden_size, n_ranks, bias=True)

    def forward_rank(self, x: torch.Tensor, rank: int) -> torch.Tensor:
        """
        Run attention for one rank (one GPU's share of heads).

        Returns the partial output for this rank's heads.
        In production, each rank runs this independently in parallel.
        """
        B, T, _ = x.shape

        # Get this rank's Q, K, V shards
        q_shard = self.q_proj.forward(x)[rank]   # (B, T, heads_per_rank * head_dim)
        k_shard = self.k_proj.forward(x)[rank]
        v_shard = self.v_proj.forward(x)[rank]

        # Reshape to (B, heads_per_rank, T, head_dim)
        q = q_shard.view(B, T, self.heads_per_rank, self.head_dim).transpose(1, 2)
        k = k_shard.view(B, T, self.heads_per_rank, self.head_dim).transpose(1, 2)
        v = v_shard.view(B, T, self.heads_per_rank, self.head_dim).transpose(1, 2)

        # Standard scaled dot-product attention (local, no communication)
        scale  = self.head_dim ** -0.5
        scores = torch.matmul(q, k.transpose(-2, -1)) * scale
        mask   = torch.tril(torch.ones(T, T, device=x.device)).bool()
        scores = scores.masked_fill(~mask.unsqueeze(0).unsqueeze(0), float('-inf'))
        attn   = F.softmax(scores, dim=-1)
        out    = torch.matmul(attn, v)   # (B, heads_per_rank, T, head_dim)

        # Flatten heads back: (B, T, heads_per_rank * head_dim)
        return out.transpose(1, 2).contiguous().view(B, T, self.heads_per_rank * self.head_dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Full parallel forward pass: run all ranks, then all-reduce output projection."""
        rank_outputs = [self.forward_rank(x, r) for r in range(self.n_ranks)]
        return self.out_proj.forward(rank_outputs)   # all-reduce


# Verify correctness against standard multi-head attention
HIDDEN, HEADS, N_RANKS, B, T = 256, 8, 4, 2, 16

tp_attn  = TensorParallelAttention(HIDDEN, HEADS, N_RANKS)
std_attn = nn.MultiheadAttention(HIDDEN, HEADS, batch_first=True, bias=True)

# Copy weights so both compute the same function
W_q = torch.cat([s.data for s in tp_attn.q_proj.weight_shards], dim=0)
W_k = torch.cat([s.data for s in tp_attn.k_proj.weight_shards], dim=0)
W_v = torch.cat([s.data for s in tp_attn.v_proj.weight_shards], dim=0)
in_proj_weight = torch.cat([W_q, W_k, W_v], dim=0)
with torch.no_grad():
    std_attn.in_proj_weight.copy_(in_proj_weight)
    W_out = torch.cat([s.data for s in tp_attn.out_proj.weight_shards], dim=1)
    std_attn.out_proj.weight.copy_(W_out)
    std_attn.out_proj.bias.copy_(tp_attn.out_proj.bias.data)

x = torch.randn(B, T, HIDDEN)
with torch.no_grad():
    out_tp,  _  = tp_attn.forward(x), None
    out_tp       = tp_attn.forward(x)
    out_std, _  = std_attn(x, x, x, need_weights=False)

max_err = (out_tp - out_std).abs().max().item()
print(f'Tensor-parallel attention correctness:')
print(f'  Max abs error vs nn.MultiheadAttention: {max_err:.2e}')
print(f'  Verified: {max_err < 1e-4}')


In [ ]:
# Communication overhead analysis and parallelism strategy comparison.
#
# The cost of tensor parallelism is the all-reduce communication between
# every layer. We model this cost analytically and show how it scales
# with the number of GPUs and the model hidden size.
#
# All-reduce cost model (ring-reduce algorithm):
#   time = 2 * (N-1)/N * (message_bytes / bandwidth)
# where N is the number of GPUs and bandwidth is the inter-GPU link speed.
#
# NVLink bandwidth (within a node): 600 GB/s (A100 NVSwitch)
# InfiniBand bandwidth (cross-node): 400 Gb/s = 50 GB/s (HDR IB)
# This 12x gap is why tensor parallelism typically stays within a single node.

def allreduce_time_ms(
    message_bytes:  int,
    n_gpus:         int,
    bandwidth_gbps: float = 600.0,   # NVLink
) -> float:
    """
    Estimate ring all-reduce latency in milliseconds.

    Ring all-reduce sends 2 * (N-1)/N * message_size bytes in total.
    Dividing by bandwidth gives wall-clock time.
    Alpha (latency) term is omitted for simplicity (negligible vs bandwidth term).
    """
    fraction = 2 * (n_gpus - 1) / n_gpus
    return fraction * message_bytes / (bandwidth_gbps * 1e9 / 1000)


# Plot communication overhead vs hidden size and GPU count
fig = plt.figure(figsize=(15, 10))
gs  = gridspec.GridSpec(2, 2, figure=fig, hspace=0.40, wspace=0.32)

# Panel 1: all-reduce time vs hidden size
ax = fig.add_subplot(gs[0, 0])
hidden_sizes = [512, 1024, 2048, 4096, 8192, 16384]
batch_sizes  = [1, 4, 8, 32]
for bs, col in zip(batch_sizes, ['#1f77b4','#ff7f0e','#2ca02c','#d62728']):
    times = [allreduce_time_ms(hs * bs * 2, n_gpus=8)  # FP16 = 2 bytes
             for hs in hidden_sizes]
    ax.plot(hidden_sizes, times, 'o-', lw=2, ms=5, color=col, label=f'batch={bs}')
ax.set_xlabel('Hidden size', fontsize=11)
ax.set_ylabel('All-reduce time (ms, NVLink 600 GB/s)', fontsize=11)
ax.set_title('All-reduce Cost vs Hidden Size\n(8 GPUs, NVLink)', fontsize=12)
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

# Panel 2: compute vs communication for LLaMA-3 8B
ax = fig.add_subplot(gs[0, 1])
gpu_counts = [1, 2, 4, 8]
# Model parameters: hidden=4096, layers=32
# Compute per decode step (memory-bandwidth bound): ~14 ms on A100 at batch=1
# All-reduce overhead: 2 all-reduces per layer * 32 layers
decode_compute_ms  = 14.0
ar_per_layer       = lambda n: allreduce_time_ms(4096 * 1 * 2, n)  # batch=1
total_comm_ms      = lambda n: 2 * 32 * ar_per_layer(n)            # 2 AR/layer
total_time         = lambda n: decode_compute_ms / n + total_comm_ms(n)

for n, col in zip(gpu_counts, ['#d62728','#ff7f0e','#2ca02c','#1f77b4']):
    compute = decode_compute_ms / n
    comm    = total_comm_ms(n)
    ax.bar(str(n), compute, color=col, alpha=0.85, label=f'{n} GPU')
    ax.bar(str(n), comm, bottom=compute, color=col, alpha=0.45)
ax.set_xlabel('Number of GPUs (tensor parallelism)', fontsize=11)
ax.set_ylabel('Time per decode step (ms, estimated)', fontsize=11)
ax.set_title('Compute vs Communication per Step\nLLaMA-3 8B, batch=1\n(solid=compute, light=comm)', fontsize=11)
ax.grid(alpha=0.3, axis='y')

# Panel 3: speedup vs number of GPUs (with communication overhead)
ax = fig.add_subplot(gs[1, 0])
gpu_range = [1, 2, 4, 8, 16]
ideal_speedup = gpu_range
real_speedup  = [1.0,
                 1.9 * 0.97,   # ~3% comm overhead at 2 GPUs
                 3.7 * 0.94,   # ~6% at 4 GPUs
                 7.0 * 0.88,   # ~12% at 8 GPUs
                 12.5 * 0.78]  # ~22% at 16 GPUs (IB bottleneck)
ax.plot(gpu_range, ideal_speedup, '--', color='gray', lw=2, label='Ideal linear')
ax.plot(gpu_range, real_speedup, 'o-', color='#2ca02c', lw=2.5, ms=8,
        label='Tensor parallel (estimated)')
for n, s in zip(gpu_range, real_speedup):
    efficiency = s / n * 100
    ax.annotate(f'{efficiency:.0f}%', (n, s), xytext=(3, 5),
                textcoords='offset points', fontsize=8, color='#2ca02c')
ax.set_xlabel('Number of GPUs', fontsize=11)
ax.set_ylabel('Throughput speedup', fontsize=11)
ax.set_title('Tensor Parallel Scaling Efficiency\n(% annotations = scaling efficiency)', fontsize=12)
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

# Panel 4: TP vs PP strategy comparison
ax = fig.add_subplot(gs[1, 1])
strategies = [
    ('TP=1, PP=1\n(single GPU)',   1,   1, '#d62728'),
    ('TP=2, PP=1\n(NVLink pair)',  1.9, 1, '#ff7f0e'),
    ('TP=4, PP=1\n(NVLink quad)',  3.5, 1, '#2ca02c'),
    ('TP=1, PP=4\n(pipeline)',     3.7, 0.7, '#1f77b4'),
    ('TP=2, PP=2\n(hybrid)',       3.6, 0.95,'#9467bd'),
]
labels    = [s[0] for s in strategies]
throughputs = [s[1] for s in strategies]
latency_factors = [s[2] for s in strategies]  # relative to TP=4
cols = [s[3] for s in strategies]
x = np.arange(len(labels))
bars = ax.bar(x, throughputs, color=cols, alpha=0.85, width=0.5)
for b, v in zip(bars, throughputs):
    ax.text(b.get_x() + b.get_width()/2, v + 0.05, f'{v}x',
            ha='center', fontsize=9, fontweight='bold')
ax2 = ax.twinx()
ax2.plot(x, latency_factors, 'D--', color='black', ms=7, lw=1.5, label='Relative latency')
ax2.set_ylabel('Relative per-request latency (lower=better)', fontsize=9)
ax2.set_ylim(0, 1.3)
ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=8)
ax.set_ylabel('Throughput speedup vs single GPU', fontsize=11)
ax.set_title('TP vs PP Strategy Comparison\n(4 GPUs, LLaMA 8B, estimated)', fontsize=12)
ax.grid(alpha=0.3, axis='y')

plt.suptitle('Tensor Parallelism and Distributed Inference', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../figures/11_tensor_parallel.png', dpi=150, bbox_inches='tight')
plt.show()


## 4. Engineering Notes

**Parallelism decision guide**

| Situation | Recommended strategy |
|-----------|---------------------|
| Model fits on 1 GPU | No parallelism |
| Model needs 2-8 GPUs, same node | Tensor parallelism (NVLink) |
| Model needs >8 GPUs | TP within node + PP across nodes |
| Optimising for latency | Tensor parallelism (no pipeline bubbles) |
| Optimising for throughput at large batch | Pipeline parallelism |

**Production parallelism with vLLM**

```python
from vllm import LLM

# 4-way tensor parallelism across 4 GPUs on the same node
llm = LLM(
    model='meta-llama/Llama-3-70B-Instruct',
    tensor_parallel_size=4,   # splits attention heads and MLP across 4 GPUs
    gpu_memory_utilization=0.90,
)

# 2 TP * 2 PP = 4 GPUs: TP within each pair, PP across pairs
llm = LLM(
    model='meta-llama/Llama-3-70B-Instruct',
    tensor_parallel_size=2,
    pipeline_parallel_size=2,
)
```

## 5. Exercises

1. **Sequence parallelism**: Implement sequence parallelism for layer norms:
   split the input token sequence across N ranks instead of the weight matrices.
   Show that this requires all-gather before and reduce-scatter after each attention.

2. **Communication profiling**: Instrument `allreduce_time_ms` to record all
   communication events during a 32-layer model decode step. Plot the timeline
   showing compute and communication intervals. What fraction of total step time
   is communication at 8 GPUs with NVLink?

3. **Expert parallelism (MoE)**: Implement a simplified Mixture-of-Experts layer
   where each expert is on a different GPU. Implement the all-to-all communication
   needed to route tokens to the correct expert GPU.

4. **Load imbalance in PP**: In pipeline parallelism, the first and last pipeline
   stages have different compute (embedding lookup, final LM head). Measure the
   imbalance by timing each stage for a 32-layer model split across 4 stages.

5. **Bandwidth measurement**: Use `torch.distributed` with NCCL to measure actual
   all-reduce bandwidth on your hardware. Compare against the theoretical NVLink
   bandwidth. What is the practical efficiency (measured / theoretical)?

---

## 6. References

- [Shoeybi et al. (2019) -- Megatron-LM: Training Multi-Billion Parameter Language Models](https://arxiv.org/abs/1909.08053)
- [Narayanan et al. (2021) -- Efficient Large-Scale Language Model Training on GPU Clusters](https://arxiv.org/abs/2104.04473)
- [Rajbhandari et al. (2020) -- ZeRO: Memory Optimizations Toward Training Trillion Parameter Models](https://arxiv.org/abs/1910.02054)
- [vLLM Distributed Inference Documentation](https://docs.vllm.ai/en/latest/serving/distributed_serving.html)
